In [3]:
setwd('/home/ec2-user/SageMaker/phenotype/input')

In [4]:
load('dat_feat_2015.Rdata')
# load('RA_label.Rdata')
# load('CHF_label.Rdata')
load('Depression_label.Rdata')
# RA is PheCode:714.1, which means Rheumatoid Arthritis(风湿性关节炎)
# CHF is PheCode:428, which means congestive heart failure(充血性心力衰竭)
# depression is PheCode:296.2

In [5]:
name_desc <- read.csv('https://han-attention.s3.amazonaws.com/input/name_desc/name_desc_all_GPT4.csv')
emb_all <- read.csv('naive_0908.csv')
emb_MGB <- emb_all[1:6917,]

In [23]:
library(purrr)

gen_pai_emb <- function(emb, dat.feat, dat.label, phecode) {
    # only select patients with freq features
    dat.label <- dat.label[which(dat.label[,1] %in% dat.feat[,1]),]
    # only select freq features with labels
    dat.feat <- dat.feat[match(dat.label[,1], dat.feat[,1]),]
    colnames(dat.feat) <- gsub('CCS-PCS', 'CCS', colnames(dat.feat))
    matching_cols <- which(colnames(dat.feat) %in% emb[,1])
    dat.feat <- dat.feat[,c(1, matching_cols, ncol(dat.feat))]
    dat.feat[,ncol(dat.feat)] <- apply(dat.feat[,2:(ncol(dat.feat)-1)], 1, sum)
    emb <- emb[match(colnames(dat.feat)[2:(ncol(dat.feat)-1)], emb[,1]),]
    emb_phe <- as.matrix(emb[match(phecode, emb[,1]), 2:769])
    emb <- t(apply(emb[,2:769],1,as.numeric))
    cos <- emb %*% t(emb_phe)

    compute_emb_row <- function(row) {
        freq_code <- as.numeric(row[2:(ncol(dat.feat) - 1)]) * cos
        return(t(emb) %*% freq_code)
    }

    # Initialize txtProgressBar
    pb <- txtProgressBar(min = 0, max = nrow(dat.feat), style = 1)
    emb_pat <- matrix(, nrow = nrow(dat.feat), ncol = ncol(emb))

    for(i in 1:nrow(dat.feat)) {
        emb_pat[i, ] <- compute_emb_row(as.numeric(dat.feat[i, ]))
        setTxtProgressBar(pb, i)  # Update progress bar
    }

    close(pb)  # Close the progress bar
    emb_pat <- apply(emb_pat, 2, function(x) x/sqrt(sum(x^2)))

    return(emb_pat)
}

In [ ]:
emb_pat <- gen_pai_emb(emb_MGB, dat.feat, dat.label, 'PheCode:714')

In [113]:
dat.label_ <- dat.label[which(dat.label[,1] %in% dat.feat[,1]),]

In [107]:
# write.csv(emb_pat,'RA_patient_emb_8_23.csv', row.names=dat.label_[,1])
# write.csv(emb_pat,'CHF_patient_emb_8_23.csv', row.names=dat.label_[,1])
write.csv(emb_pat,'depression_patient_emb_8_23.csv', row.names=dat.label_[,1])

In [ ]:
# dat.label <- dat.label[which(dat.label[,1] %in% dat.feat[,1]),]

# rank_visualization<-function(data, group, test)
# {
#   mylevel<-unique(group)
#   r<-length(mylevel)
  
#   pos<-vector("list",r)
  
#   for(i in 1:r)
#   {
#     pos[[i]]<-(group==mylevel[i])
#   }
  
#   Z<-data
#   N<-dim(data)[1]
  
#   D<-as.matrix(dist(Z,method = "euclidean"))
#   RG<-matrix(rep(0,N^2),nrow = N)
  
#   for(i in 1:N)
#   {
#     RG[i,]<-rank(D[i,])
#   }
  
#   library(MASS)
  
#   M<-matrix(NA,nrow=N,ncol=r)
  
#   for(i in 1:N)
#   {
#     for(j in 1:r)
#     {
#       a<-RG[pos[[j]],i]
#       M[i,j]<-mean(a[a!=0])
#     }
#   }
  
#   lw<-dim(test)[1]
#   MW<-matrix(NA,nrow = lw,ncol = r)
#   for(k in 1:lw)
#   {
#     w<-test[k,]
    
#     dw<-rep(0,N)
    
#     for(i in 1:N)
#     {
#       dw[i]<-norm(w-data[i,],type="2")
#     }
    
#     rw<-rep(0,N)
    
#     for(i in 1:N)
#     {
#       rw[i]<-sum(D[i,]<dw[i])+0.5
#     }
    
#     Mw<-rep(NA,r)
    
#     for(i in 1:r)
#     {
#       Mw[i]<-mean(rw[pos[[i]]])
#     }
    
#     MW[k,]<-Mw
#   }
  
#   ans <- list(M,MW)
#   names(ans) <- c("train_k_dimension","test_k_dimension")
#   return(ans)
# }

# prepare_data <- function(emb_pat, dat.feat, dat.label, Y_num, N_num){
#     group <- dat.label[,2]
#     Y <- which(group == "Y")
#     N <- which(group == "N")
#     Y_train <- sample(Y, Y_num)
#     N_train <- sample(N, N_num)
#     Y_test <- setdiff(Y, Y_train)
#     N_test <- setdiff(N, N_train)
#     group_train <- c(rep(1, Y_num), rep(2, N_num))
#     group_test <- c(rep(1, length(Y_test)), rep(2, length(N_test)))
#     data <- emb_pat[c(Y_train, N_train),]
#     test <- emb_pat[c(Y_test, N_test),]
#     return(list(data=data, group_train=group_train, test=test, group_test=group_test))
# }

# prepare_qda <- function(data, group_train){
#     mu1 = apply(data[group_train == 1,], 2, mean)
#     mu2 = apply(data[group_train == 2,], 2, mean)
#     Sigma1 = cov(data[group_train == 1,])
#     Sigma2 = cov(data[group_train == 2,])
#     prior1 = sum(group_train == 1)/length(group_train)
#     prior2 = sum(group_train == 2)/length(group_train)
#     return(list(mu1=mu1, mu2=mu2, Sigma1=Sigma1, Sigma2=Sigma2, prior1=prior1, prior2=prior2 ))
# }

# qda_predict_binary <- function(new_data, mu1, mu2, Sigma1, Sigma2, prior1, prior2) {
  
#   # 计算逆协方差矩阵
#   inv_Sigma1 <- solve(Sigma1)
#   inv_Sigma2 <- solve(Sigma2)
  
#   # 对于每个类别计算分数
#   score1 <- vector("numeric", length = nrow(new_data))
#   score2 <- vector("numeric", length = nrow(new_data))
  
#   for (i in 1:nrow(new_data)) {
#     x <- as.numeric(new_data[i,])    
#     score1[i] <-log(prior1) - 0.5 * log(det(Sigma1)) -0.5 * t(mu1) %*% inv_Sigma1 %*% mu1 + t(mu1) %*% inv_Sigma1 %*% x -0.5 * t(x) %*% inv_Sigma1 %*% x
    
#     term1_class2 <- log(prior2) - 0.5 * log(det(Sigma2))
#     term2_class2 <- -0.5 * t(mu2) %*% inv_Sigma2 %*% mu2
#     term3_class2 <- t(mu2) %*% inv_Sigma2 %*% x
#     term4_class2 <- -0.5 * t(x) %*% inv_Sigma2 %*% x
    
#     score2[i] <- term1_class2 + term2_class2 + term3_class2 + term4_class2
#   }
  
#   # 比较两个类别的分数
#   predicted_class <- ifelse(score1 > score2, 1,2)
  
#   return(predicted_class)
# }


# install.packages('pROC')

# library(pROC)

# total_rand_phenptyping <- function(emb_pat, dat.feat, dat.label, Y_num, N_num){
#     lis = prepare_data(emb_pat, dat.feat, dat.label, Y_num, N_num)
#     ans = rank_visualization(lis$data, lis$group_train, lis$test)
#     qda_lis = prepare_qda(ans$train_k_dimension, lis$group_train)
#     predictions <- qda_predict_binary(ans$test_k_dimension, qda_lis$mu1, qda_lis$mu2, 
#                                       qda_lis$Sigma1, qda_lis$Sigma2, qda_lis$prior1, qda_lis$prior2)
#     roc_obj <- roc(lis$group_test,predictions)
#     return(auc(roc_obj))
# }


# predictions <- total_rand_phenptyping(emb_pat, dat.feat, dat.label, 10,10)

# predictions

